# CrossFuse v5 -- Stage B0: Self-Blended Images Visual Pretraining

Trains an EfficientNet-B4 encoder on FF++ real-video frames only, using the
full self-blend recipe in `sbi_v5.py` (landmark-driven mask, source/target
transforms, elastic mask deformation, random blend ratio). No FF++ *fakes*
and no FakeAVCeleb data are used here -- that is the point: the resulting
encoder learns the blending-boundary / resolution-mismatch artifact that
every face-swap method leaves, rather than one generator's fingerprint.

**Verification gate (do not skip):** the paper (arXiv:2204.08376) reports
93.18 AUC on Celeb-DF for this exact recipe. The last cell here runs a
zero-shot check against a small Celeb-DF sample. If it does not clear 0.85,
something in the SBI implementation is wrong -- check the
`[sbi] landmark tier usage, epoch 1 train pass (...)` line printed during
training FIRST: if `parametric` dominates that count, landmark detection
degraded silently (MediaPipe and MTCNN both failed) and the encoder should
not be trusted regardless of its self-blend AUC. That line is aggregated
from actual DataLoader worker output and is the one to trust -- the
`landmark_backend()` print below only reflects the main process, not the
workers where landmark detection actually runs. Do NOT proceed to load this
checkpoint into NB-B until the gate clears.

Attach as input: `ffpp-real-crops-v5` (from NB-A2), `crossfuse-v5-lib`, and a
small Celeb-DF v2 sample dataset for the gate check.


In [ ]:
!pip install -q --no-deps facenet-pytorch
!pip install -q mediapipe
import sys, glob, os
_lib_hits = glob.glob("/kaggle/input/**/crossfuse_v5.py", recursive=True)
assert _lib_hits, ("crossfuse_v5.py not found under /kaggle/input -- attach the "
                   "crossfuse-v5-lib dataset in Add Input, then restart the session.")
sys.path.insert(0, os.path.dirname(_lib_hits[0]))
print("Using crossfuse_v5 library from:", _lib_hits[0])

import os, glob
import numpy as np
import torch

from sbi_v5 import (landmark_backend, train_sbi_encoder, SBIClassifier,
                    score_frames_video_level)
from crossfuse_v5 import build_detector, extract_face_crops_contiguous, safe_auc, make_rng

print(f"Landmark backend: {landmark_backend()}  "
      f"(must be 'mediapipe' for a real SBI run -- 'parametric-fallback' means "
      f"MediaPipe failed to import and results below should not be trusted as SBI's)")


In [ ]:
# ---- 1. Build the frame index from NB-A2's cache ----
crop_files = sorted(glob.glob("/kaggle/input/**/ffppreal_*.npy", recursive=True))
assert crop_files, "No ffpp-real-crops-v5 files found -- attach NB-A2's published dataset."
print(f"{len(crop_files)} cached videos found")

index = []
for p in crop_files:
    n = np.load(p, mmap_mode="r").shape[0]
    index.extend([(p, i) for i in range(n)])
print(f"{len(index)} frames total")

rng = make_rng("sbi_video_split")
video_ids = list(range(len(crop_files)))
rng.shuffle(video_ids)
n_val_videos = max(1, int(0.1 * len(video_ids)))
val_video_set = set(video_ids[:n_val_videos])
train_index = [(p, i) for p, i in index
              if crop_files.index(p) not in val_video_set]
val_index = [(p, i) for p, i in index if crop_files.index(p) in val_video_set]
print(f"train frames: {len(train_index)} | val frames: {len(val_index)} "
      f"(split at VIDEO level, not frame level, to avoid the same identity "
      f"leaking across the split)")


In [ ]:
OUT_CKPT = "/kaggle/working/sbi_visual_encoder_v5.pth"
# multi_gpu=False: DataParallel crashed a live run (kernel died) at
# efficientnet_b4 + res 380 + effective batch 32 -- its master GPU gathers
# every replica's output for loss/gradient reduction and carries more than
# an even memory share. precompute_lm=True is kept: it ran cleanly (7981
# mtcnn5 / 19 parametric on 8000 frames) and is what actually fixes the
# CPU-bound loader (previously observed GPU0 ~30%, GPU1 0%).
result = train_sbi_encoder(
    train_index, val_index, OUT_CKPT,
    backbone="efficientnet_b4", out_size=380, epochs=8, batch_size=16,
    lr=1e-4, num_workers=4, seed=42,
    precompute_lm=True, multi_gpu=False)
print(result)


In [ ]:
# ---- 2. Verification gate: zero-shot Celeb-DF AUC must clear 0.85 ----
celebdf_videos = (glob.glob("/kaggle/input/**/Celeb-real/**/*.mp4", recursive=True) +
                  glob.glob("/kaggle/input/**/Celeb-synthesis/**/*.mp4", recursive=True))
if not celebdf_videos:
    print("[!] No Celeb-DF sample attached -- cannot run the verification gate. "
          "Attach a Celeb-DF v2 sample dataset and re-run this cell before trusting "
          "this encoder in NB-B.")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    clf = SBIClassifier(backbone="efficientnet_b4", pretrained=False).to(device)
    ckpt = torch.load(OUT_CKPT, map_location=device, weights_only=False)
    clf.backbone.load_state_dict(ckpt["backbone_state"])
    clf.eval()

    detector = build_detector(str(device))
    N_PER_CLASS = 60
    rng = make_rng("sbi_gate")
    rng.shuffle(celebdf_videos)

    y_true, y_score = [], []
    n_real, n_fake = 0, 0
    for vpath in celebdf_videos:
        label = 0 if "celeb-real" in vpath.lower() or "youtube-real" in vpath.lower() else 1
        if (label == 0 and n_real >= N_PER_CLASS) or (label == 1 and n_fake >= N_PER_CLASS):
            continue
        crops, valid, *_ = extract_face_crops_contiguous(
            vpath, detector, rng=make_rng(vpath), n_frames=16, out_size=256)
        if crops is None or int(valid.sum()) == 0:
            continue
        score = score_frames_video_level(clf, crops[valid], device=device, out_size=380)
        y_true.append(label); y_score.append(score)
        n_real += (label == 0); n_fake += (label == 1)
        if n_real >= N_PER_CLASS and n_fake >= N_PER_CLASS:
            break

    auc = safe_auc(y_true, y_score)
    print(f"\nZero-shot Celeb-DF AUC: {auc:.4f}  (n={len(y_true)}, "
          f"{n_real} real / {n_fake} fake)")
    if auc >= 0.85:
        print("[OK] Gate cleared -- proceed to NB-B with this checkpoint.")
    elif auc <= 0.45:
        # Sub-chance is NOT the same failure as near-chance and must not be
        # read as "no signal". It means the encoder ranks Celeb-DF reals as
        # MORE fake than its fakes -- a real, strong, INVERTED cue. That is
        # the signature of a shortcut learned from the synthetic data whose
        # direction happens to flip on this dataset (the shipped v1 scored
        # 0.38 this way, by learning "blurry => fake" from an asymmetric
        # blend; Celeb-DF's reals are soft YouTube uploads and its v2 fakes
        # are deliberately high-resolution, so the rule reverses).
        print(f"[FAIL-INVERTED] AUC {auc:.4f} is BELOW chance; flipping the sign "
              f"would give {1 - auc:.4f}. This is a shortcut pointing the wrong "
              f"way, not an absence of signal. Do NOT 'fix' it by inverting the "
              f"score -- find the statistic that leaked. Check whether real and "
              f"fake training samples differ in any GLOBAL property (sharpness, "
              f"brightness, compression) rather than only at the blend boundary.")
    else:
        print(f"[FAIL] AUC {auc:.4f} is near chance -- the encoder found little "
              f"transferable signal. Scroll up to the training cell's '[sbi] "
              f"landmark tier usage, epoch 1 train pass (...)' line: if "
              f"'parametric' dominates, landmark detection degraded silently "
              f"(restart the session to clear stale cached failure state). If "
              f"mediapipe/mtcnn5 dominates and this still fails, check that "
              f"CROP_SIZE/margin in NB-A2 left enough context around the face "
              f"for the blend mask.")
